# **Subjektivní analýza novinových článků**

Tématem projektu do Text Miningu jsme si zvolili **Subjektivní analýzu novinových článků**.

Cílem je zjistit zda dokážeme čerpat z novinových článků informace o zaujatosti autora. Druhotným cílem je znázornit možné cesty a variace jak výsledku dosáhnout nebo jej vylepšit pomocí technik používaných v oboru text miningu.

### Stažení potřebných knihoven.
Nechali jsme si doporučit knihovny hodící se k extrahování informací z textu a webovému scrapingu. Mimo to jsme použili doporučený Hugging Face z cvičení z důvodu obsahování pokročilých modelů.
Jsou jimi:
- **newspaper3k**: Knihovna pro stahování, parsování a analýzu novinových článků a blogů z webu.
- **lxml_html_clean**: Nástroj pro čištění HTML dokumentů, odstranění nepotřebných tagů a atributů.
- **textblob**: Knihovna pro zpracování textu, která poskytuje jednoduché API pro běžné úlohy NLP, jako je značkování části řeči, extrakce podstatných jmen, analýza sentimentu a další.
- **transformers**: Knihovna od Hugging Face pro práci s nejmodernějšími předtrénovanými modely (jako BERT, GPT-2 atd.) pro různé NLP úlohy.
- **torch**: Open-source knihovna pro strojové učení, která se používá pro tvorbu a trénování neuronových sítí.
- **ufal.udpipe**: Nástroj pro zpracování přirozeného jazyka (NLP), který poskytuje tokenizaci, lemmatizaci, značkování části řeči (POS tagging) a syntaktickou analýzu pro mnoho jazyků, včetně češtiny.
- **requests**: Jednoduchá HTTP knihovna pro Python, která umožňuje snadno odesílat HTTP požadavky.

In [ ]:
!pip install newspaper3k lxml_html_clean textblob transformers torch ufal.udpipe requests

: 

### 1. & 2. Sběr a extrakce textu z článků
Pro scrapování lze použít knihovna `newspaper3k`, která automaticky detekuje text článku, autora i datum publikace. Bohužel pro české články to ne vždy platí a tak je třeba navést tuto knihovnu ručně na tagy ve stránce.

**Deginice funkce**

In [ ]:
from newspaper import Article, build, Config
import requests
from bs4 import BeautifulSoup
import json

def get_articles_from_url(base_url, num_articles=None):
    config = Config()
    config.browser_user_agent = 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/119.0.0.0 Safari/537.36'
    config.request_timeout = 30

    print(f"Pokouším se stáhnout články z: {base_url}")
    try:
        paper = build(base_url, config=config, memoize_articles=False)
        print(f"Nalezeno {len(paper.articles)} potenciálních článků.")
    except Exception as e:
        print(f"Chyba při načítání stránek nebo budování zdroje: {e}")
        return []

    articles_data = []
    filtered_articles = [article_obj for article_obj in paper.articles if '/clanek/' in article_obj.url]
    print(f"Po filtrování podle '/clanek/' zbývá {len(filtered_articles)} potenciálních článků.")

    for i, article_obj in enumerate(filtered_articles):
        if num_articles is not None and i >= num_articles:
            break
        try:
            article = Article(article_obj.url, config=config)
            article.download()
            article.parse()

            soup = None
            if article.html:
                soup = BeautifulSoup(article.html, 'lxml')

            # --- EXTRAKCE TITULKU ---
            if not article.title and soup:
                og_title = soup.find('meta', property='og:title')
                if og_title and og_title.get('content'):
                    article.title = og_title['content'].strip()
                elif soup.title:
                    article.title = soup.title.get_text(strip=True).replace(' - Seznam Zprávy', '')
                else:
                    h1_tag = soup.find('h1')
                    if h1_tag:
                        article.title = h1_tag.get_text(strip=True)

            # --- EXTRAKCE AUTORA A DATA PUBLIKACE (PŘES JSON-LD) ---
            authors_list = article.authors
            publish_date = None # Začínáme s None, chceme ideálně přesný ISO string z HTML

            if soup:
                for script in soup.find_all('script', type='application/ld+json'):
                    try:
                        data = json.loads(script.string)
                        if isinstance(data, list):
                            data = data[0] # Vezmeme první prvek, pokud jde o seznam

                        # Hledáme schema.org typy pro články
                        if isinstance(data, dict) and data.get('@type') in ['NewsArticle', 'Article', 'ReportageNewsArticle']:

                            # Vytáhneme datum
                            if not publish_date and data.get('datePublished'):
                                publish_date = data.get('datePublished')

                            # Vytáhneme autory
                            if not authors_list:
                                author_data = data.get('author')
                                if isinstance(author_data, list):
                                    authors_list = [a.get('name') for a in author_data if isinstance(a, dict) and a.get('name')]
                                elif isinstance(author_data, dict) and author_data.get('name'):
                                    authors_list = [author_data.get('name')]
                    except (json.JSONDecodeError, TypeError, AttributeError):
                        continue

            # --- ZÁLOŽNÍ EXTRAKCE DATA PUBLIKACE (Meta tagy) ---
            if not publish_date and soup:
                # Nejspolehlivější meta tag pro zpravodajské weby
                meta_time = soup.find('meta', property='article:published_time')
                if meta_time and meta_time.get('content'):
                    publish_date = meta_time['content'].strip()
                else:
                    # Klasický schema.org meta tag
                    meta_date = soup.find('meta', attrs={'itemprop': 'datePublished'})
                    if meta_date and meta_date.get('content'):
                        publish_date = meta_date['content'].strip()

            # Fallback na původní knihovnu newspaper3k (kdyby všechno ostatní selhalo)
            if not publish_date and article.publish_date:
                # Newspaper3k často vrací 'datetime' objekt. Převedeme ho na text, ať je to konzistentní
                publish_date = article.publish_date.strftime('%Y-%m-%dT%H:%M:%S')

            # --- ZÁLOŽNÍ EXTRAKCE AUTORA (Meta tag) ---
            if not authors_list and soup:
                meta_author = soup.find('meta', attrs={'name': 'author'})
                if meta_author and meta_author.get('content'):
                    authors_list = [meta_author['content'].strip()]


            # --- UKLÁDÁNÍ VÝSLEDKŮ ---
            if article.text and article.title:
                articles_data.append({
                    'url': article.url,
                    'title': article.title,
                    'authors': authors_list,
                    'publish_date': publish_date, # Nyní spolehlivě v ISO 8601 formátu
                    'text': article.text
                })
                print(f"Stažen článek {i+1}: {article.title} | Datum: {publish_date} | Autoři: {', '.join(authors_list) if authors_list else 'Neznámý'}")
            else:
                print(f"Článek {i+1} z {article_obj.url} neobsahuje text nebo titulek, přeskočen.")

        except Exception as e:
            print(f"Chyba při stahování nebo parsování článku z {article_obj.url}: {e}")
            continue

    return articles_data

**Spuštění funkce** pro scrapování článků z www.seznamzpravy.cz/

- Článek od článku dochází k přehlédnutí nadpisu a tím dojde k zahození článků. Tato chybavost nám ale nevadí, vždy můžem zvednout počet článků, kdybychom potřebovali více textu na dedukci.
- Z článků extrahujeme [odkaz, název článku, autora a datum publikace]

In [ ]:
import pandas as pd

# For example, to download 10 articles:
num_articles_to_download = 20
seznam_zpravy_articles = get_articles_from_url("https://www.seznamzpravy.cz/", num_articles=num_articles_to_download)

if seznam_zpravy_articles:
    articles_df = pd.DataFrame(seznam_zpravy_articles)
    display(articles_df[['url', 'title', 'authors', 'publish_date']].head(num_articles_to_download))
else:
    print("Nebyly staženy žádné články nebo došlo k chybě.")

### 3. Předzpracování textu
Úspěšně jsme extrahovali potřebná data pro analýzu článků. V této části preprocesujeme data pomocí zmíňených postupů z vyuky. Snažíme se text očistit takovým způsobem, aby jsme se zbavili nepotřebných informací a zvýšili užitnou hodnotu pro náš cíl.
Postupy které jsme zvolili pro náš cíl jsou:
1.  **Převod na malá písmena**: Sjednotí všechna slova, takže "Slovo" a "slovo" jsou považovány za stejné. To je klíčové pro správné počítání frekvence slov a pro konsistentní analýzu.
2.  **Odstranění interpunkce**: Interpunkční znaménka často nepřinášejí sémantický význam pro analýzu textu a mohou zkreslovat výsledky (např. "slovo." vs. "slovo"). Jejich odstranění zjednodušuje text a zlepšuje rozpoznávání slov.
3.  **Odstranění číslic** <--- kontroverzní krok: Číslice (např. data, množství) mohou v některých kontextech být důležité, ale pro analýzu zaujatosti nebo sémantickou podobnost často představují šum.
4.  **Odstranění stop slov**: Stop slova (jako "a", "je", "na") jsou velmi častá, ale obvykle nepřinášejí mnoho významových informací. Jejich odstranění snižuje dimenzionalitu dat a pomáhá zaměřit se na důležitější slova, která lépe odrážejí obsah textu.
5.  **Odstranění přebytečných mezer**: Vícenásobné mezery mohou vzniknout po odstranění jiných prvků a mohou způsobovat problémy při tokenizaci nebo dalších fázích zpracování. Jejich sjednocení na jednu mezeru udržuje text čistý a formátovaný.

**Definice funkcí:**

In [ ]:
import re
import string

# Česká stop slova - rozšířený seznam
czech_stop_words = set([
    "a", "aby", "ahoj", "ale", "anebo", "ani", "ano", "asi", "ať", "bez", "bude", "budete", "budeme", "budeš", "budou", "by", "byl", "byla", "byli", "bylo", "být", "či", "co", "což", "další", "díky", "dle", "dnes", "do", "dobrý", "docela", "dost", "eventuálně", "furt", "hodně", "já", "jak", "jako", "je", "jeden", "jednak", "jednou", "jej", "jejich", "její", "jemu", "jen", "jenom", "jestli", "jestliže", "jež", "ji", "jinak", "jiný", "již", "jí", "jsem", "jsi", "jsme", "jsou", "jste", "k", "kam", "kde", "kdo", "kdy", "když", "ke", "kolik", "kromě", "která", "které", "který", "kteří", "ktera", "kterou", "kterými", "ktery", "kvůli", "ma", "má", "mají", "mám", "máme", "máte", "mezi", "mě", "mně", "mnohý", "možná", "moji", "moja", "moje", "my", "na", "nad", "nám", "například", "náš", "naši", "naše", "ně", "nějaký", "někde", "někdo", "neměli", "není", "než", "nic", "nich", "ním", "niz", "nové", "nový", "o", "od", "on", "ona", "ono", "oni", "ony", "pak", "po", "pod", "podle", "pokud", "pouze", "právě", "pro", "proč", "prosím", "první", "před", "přes", "při", "přitom", "ráda", "se", "si", "sice", "spíše", "sta", "stále", "své", "svou", "svůj", "svých", "tak", "taky", "tam", "tamhle", "tato", "teď", "tehdy", "ten", "tento", "této", "tím", "tipy", "to", "tohle", "toho", "toto", "třeba", "tu", "tuto", "ty", "týden", "už", "v", "vám", "váš", "vaše", "ve", "velmi", "vedle", "více", "vlastně", "však", "všechno", "vy", "z", "za", "zatímco", "zde", "ze", "že", "či", "aby", "kdyz", "nez", "pokud", "takze", "taky", "uz", "vam", "vas", "byt", "coz", "ktera", "kterej", "kteri", "ktere", "ktery", "muj", "muze", "mych", "neznamy", "nic", "podle", "pres", "proto", "rovnez", "sice", "tak", "taky", "ten", "to", "tento", "treba", "uz", "vas", "vice", "vubec", "za", "zase", "zde", "ze"
])

def to_lowercase(text):
    """Převede text na malá písmena."""
    return text.lower()

def remove_punctuation(text):
    """Odstraní interpunkci z textu."""
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_numbers(text):
    """Odstraní číslice z textu."""
    return re.sub(r'\d+', '', text)

def remove_stopwords(text, stop_words):
    """Odstraní stop slova z textu."""
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

def remove_extra_whitespace(text):
    """Odstraní vícenásobné mezery a ořízne text."""
    return re.sub(r'\s+', ' ', text).strip()

def preprocess_text(text, stop_words):
    """Provede kompletní předzpracování textu."""
    text = to_lowercase(text)
    text = remove_punctuation(text)
    text = remove_numbers(text)
    text = remove_stopwords(text, stop_words)
    text = remove_extra_whitespace(text)
    return text

**Spuštění funkcí:**

In [ ]:
# Aplikujeme předzpracování na texty v DataFrame
if 'articles_df' in locals() and not articles_df.empty:
    articles_df['cleaned_text'] = articles_df['text'].apply(lambda x: preprocess_text(x, czech_stop_words))
    print("Předzpracování textu dokončeno. Zde je srovnání původního a vyčištěného textu pro první 3 články:")
    for i in range(min(3, len(articles_df))):
        print(f"\n--- Článek {i+1} ---")
        print(f"Původní: {articles_df.loc[i, 'text'][:300]}...") # Zobrazíme jen prvních 300 znaků
        print(f"Vyčištěný: {articles_df.loc[i, 'cleaned_text'][:300]}...")
else:
    print("DataFrame 'articles_df' neexistuje nebo je prázdný. Ujistěte se, že jste stáhli články dříve.")

### 4. Part-of-Speech Tagging (POS Tagging) pomocí Stanza
Je třeba označit slova jejich význami, aby jsme měli větší pravděpodobnost při analýze, že správně pochopíme kontext. K tomu se nám skvěle hodí POS tagging z knihovny Stanza.

K tomu, ale musíme stáhnout jejích natrénovaný model X verze ručně z webu a naimportovat jej pro naše užití.

Stanze si při POS taggingu provádí **tokenizaci, lemmatizaci a nakonec zmíněný POS** tagging sama.

**Import Stanzy:**

In [ ]:
!pip install stanza

**Stažení jejich modelu pro český jazyk:**

In [ ]:
import stanza
import pandas as pd

# Stáhneme český model (pokud ještě není stažen)
stanza.download('cs')

# Inicializace pipeline pro češtinu
nlp = stanza.Pipeline('cs', processors='tokenize,lemma,pos')

def stanza_process_text(text):
    """Provede analýzu textu pomocí Stanza a vrátí POS tagy, lemmata a přídavná jména."""
    if not text or pd.isna(text):
        return [], "", []

    doc = nlp(text)
    pos_tags = []
    lemmas = []
    adjectives = []

    for sentence in doc.sentences:
        for word in sentence.words:
            pos_tags.append((word.text, word.upos))
            lemmas.append(word.lemma)
            if word.upos == 'ADJ':
                adjectives.append(word.lemma)

    return pos_tags, " ".join(lemmas), adjectives

if 'articles_df' in locals() and not articles_df.empty:
    print("Provádím analýzu textu pomocí Stanza (to může trvat déle)... ")

    # Aplikujeme Stanza na vyčištěný text
    results = articles_df['cleaned_text'].apply(stanza_process_text)

    # Uložení výsledků do DataFrame
    articles_df['pos_tags'] = [res[0] for res in results]
    articles_df['lemmatized_text'] = [res[1] for res in results]
    articles_df['adjectives_only'] = [res[2] for res in results]

    print("Analýza dokončena.")
    display(articles_df[['title', 'pos_tags', 'lemmatized_text']].head(5))
else:
    print("DataFrame 'articles_df' neexistuje nebo je prázdný.")

______________________________________________________


### 6. Analýza TF-IDF (Term Frequency-Inverse Document Frequency)

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

if 'articles_df' in locals() and not articles_df.empty:
    # Inicializace TF-IDF vektorizéru
    # Použijeme vyčištěný text, který už nemá stop slova a interpunkci
    tfidf = TfidfVectorizer(max_features=1000)
    tfidf_matrix = tfidf.fit_transform(articles_df['cleaned_text'])

    # Získání názvů vlastností (slov)
    feature_names = tfidf.get_feature_names_out()

    def get_top_tf_idf_words(row_index, top_n=10):
        """Vrátí top N slov s nejvyšším TF-IDF skóre pro daný řádek (článek)."""
        row = tfidf_matrix.getrow(row_index).toarray().flatten()
        top_indices = row.argsort()[-top_n:][::-1]
        return [(feature_names[i], round(row[i], 4)) for i in top_indices if row[i] > 0]

    print("Top slova podle TF-IDF pro první 3 články:\n")
    for i in range(min(3, len(articles_df))):
        print(f"--- Článek {i+1}: {articles_df.loc[i, 'title']} ---")
        top_words = get_top_tf_idf_words(i)
        for word, score in top_words:
            print(f"  {word}: {score}")
        print()
else:
    print("Dataframe 'articles_df' neexistuje nebo je prázdný.")

### 7. Extrakce hlavních témat (postavená pouze na TF-IDF a 1-grammech)
Pomocí dříve vypočítaného TF-IDF skóre nyní extrahujeme pro každý článek několik nejvýznamnějších slov, která slouží jako identifikátory témat.

In [ ]:
if 'articles_df' in locals() and not articles_df.empty:
    def extract_article_topics(row_index, top_n=5):
        """Vrátí řetězec top N klíčových slov jako téma."""
        top_words_with_scores = get_top_tf_idf_words(row_index, top_n=top_n)
        return ", ".join([word for word, score in top_words_with_scores])

    # Aplikujeme na všechny články
    articles_df['top_keywords'] = [extract_article_topics(i) for i in range(len(articles_df))]

    print("Ukázka extrahovaných témat pro články:")
    display(articles_df[['title', 'top_keywords']].head(10))
else:
    print("Dataframe 'articles_df' neexistuje.")

### 6-a. Zpřesnění témat (Lemmatizace + Bigramy)
Nyní provedeme proces znovu, ale tentokrát:
1. **Zlemmatizujeme** texty pomocí UDPipe (převedeme slova na základní tvary).
2. Nastavíme TF-IDF tak, aby hledalo i **dvouslovná spojení** (ngram_range=(1, 2)).

In [ ]:
import ufal.udpipe

def get_lemmas(tokens, pipeline):
    """Extrahuje lemmata z tokenů pomocí UDPipe."""
    text = " ".join(tokens)
    processed_text = pipeline.process(text)
    conllu_input = ufal.udpipe.InputFormat.newInputFormat("conllu")
    conllu_input.setText(processed_text)

    sentence = ufal.udpipe.Sentence()
    lemmas = []
    while conllu_input.nextSentence(sentence):
        for token in sentence.words:
            if token.id > 0:
                lemmas.append(token.lemma.lower())
    return " ".join(lemmas)

if "articles_df" in locals() and not articles_df.empty:
    if "lemmatized_text" in articles_df.columns:
        print("Lemmatizace již proběhla pomocí Stanza, přeskakuji UDPipe krok.")
    else:
        try:
            print("Provádím lemmatizaci textů pomocí UDPipe...")
            articles_df["lemmatized_text"] = articles_df["cleaned_text"].str.split().apply(
                lambda x: get_lemmas(x, process_pipeline)
            )
            print("Lemmatizace dokončena.")
        except Exception as e:
            print(f"Chyba při UDPipe lemmatizaci: {e}")


In [ ]:
if 'lemmatized_text' in articles_df.columns:
    # Vektorizér s podporou dvouslovných spojení (ngram_range)
    tfidf_refined = TfidfVectorizer(max_features=1000, ngram_range=(1, 2))
    tfidf_matrix_refined = tfidf_refined.fit_transform(articles_df['lemmatized_text'])
    feature_names_refined = tfidf_refined.get_feature_names_out()

    def get_refined_topics(row_index, top_n=5):
        row = tfidf_matrix_refined.getrow(row_index).toarray().flatten()
        top_indices = row.argsort()[-top_n:][::-1]
        return ", ".join([feature_names_refined[i] for i in top_indices if row[i] > 0])

    articles_df['top_keywords_refined'] = [get_refined_topics(i) for i in range(len(articles_df))]

    print("Srovnání: Původní vs. Zpřesněná témata (s lemmatizací a frázemi):")
    display(articles_df[['title', 'top_keywords', 'top_keywords_refined']].head(10))

### 9. Sémantická vektorizace (Word2Vec) - PŘESKOČIT
Na rozdíl od TF-IDF, které řeší pouze výskyt, Word2Vec vytváří husté vektory (embeddings). Pokud se slova v textu vyskytují v podobných kontextech, budou mít podobné vektorové vyjádření.

In [ ]:
!pip install gensim

In [ ]:
from gensim.models import Word2Vec

if 'articles_df' in locals() and 'lemmatized_text' in articles_df.columns:
    # Příprava dat: Word2Vec očekává seznam seznamů slov (tokenů)
    sentences = [text.split() for text in articles_df['lemmatized_text']]

    # Trénování modelu (pro malý počet článků je to spíše ukázka)
    # vector_size: rozměr vektoru, window: kontextové okno, min_count: ignorovat vzácná slova
    w2v_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

    print("Model Word2Vec byl natrénován.")

    # Ukázka: Najdeme slova podobná vybranému klíčovému slovu (pokud existuje v modelu)
    # Zkusíme najít top 10 slov z TF-IDF a podívat se na jejich sousedy
    example_word = feature_names_refined[3]

    try:
        similar_words = w2v_model.wv.most_similar(example_word, topn=5)
        print(f"\nSlova sémanticky nejpodobnější slovu '{example_word}':")
        for word, score in similar_words:
            print(f"  {word}: {round(score, 4)}")
    except KeyError:
        print(f"\nSlovo '{example_word}' není v modelu dostatečně zastoupeno.")
else:
    print("Chybí lemmatizovaná data pro Word2Vec.")

### 10. Analýza sentimentu a subjektivity (Zaujatost)
Pro detekci zaujatosti použijeme český model z knihovny `transformers`. Tento model nám řekne, zda je text laděn pozitivně, nebo negativně. K tomu přidáme analýzu subjektivity.

In [ ]:
from transformers import pipeline
import pandas as pd
import os

try:
    # Načtení tokenu z prostředí (místo Colab Secrets)
    hf_token = os.environ.get('HF_TOKEN')

    # Načtení vícejazyčného modelu pro analýzu sentimentu s použitím tokenu
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model="nlptown/bert-base-multilingual-uncased-sentiment",
        token=hf_token
    )

    def analyze_bias(text):
        # Transformery mají limit na délku textu, omezíme tedy vstup
        truncated_text = text[:1500]
        try:
            result = sentiment_pipeline(truncated_text)[0]
            return result['label'], round(result['score'], 4)
        except Exception as e:
            return "Error", 0.0

    if 'articles_df' in locals() and not articles_df.empty:
        print("Provádím analýzu sentimentu článků pomocí vícejazyčného modelu...")
        results = articles_df['text'].apply(analyze_bias).tolist()
        articles_df[['sentiment_label', 'sentiment_score']] = pd.DataFrame(results, index=articles_df.index)

        print("Analýza dokončena. Výsledky (1-5 hvězdiček):")
        display(articles_df[['title', 'sentiment_label', 'sentiment_score']].head(10))
    else:
        print("Dataframe 'articles_df' neexistuje.")
except Exception as e:
    print(f"Došlo k chybě při inicializaci modelu: {e}")
    print("Pokud chcete použít HuggingFace model, nastavte proměnnou prostředí HF_TOKEN.")


### 12. Výpočet vlastního Indexu subjektivity
Na základě lingvistické analýzy (POS tagging) určíme poměr modifikátorů (přídavná jména, příslovce) k faktickým nositelům děje (podstatná jména, slovesa).

In [ ]:
def calculate_subjectivity_index(pos_tags):
    """Vypočítá index subjektivity na základě poměru slovních druhů."""
    if not pos_tags:
        return 0.0

    counts = {'ADJ': 0, 'ADV': 0, 'NOUN': 0, 'VERB': 0}

    for word, tag in pos_tags:
        if tag in counts:
            counts[tag] += 1

    # Jmenovatel (fakta): Podstatná jména + Slovesa
    factual_base = counts['NOUN'] + counts['VERB']
    # Čitatel (emoce/popis): Přídavná jména + Příslovce
    descriptive_part = counts['ADJ'] + counts['ADV']

    if factual_base == 0:
        return 0.0

    # Výpočet indexu
    index = descriptive_part / factual_base
    return round(index, 4)

if 'articles_df' in locals() and 'pos_tags' in articles_df.columns:
    articles_df['subjectivity_index'] = articles_df['pos_tags'].apply(calculate_subjectivity_index)

    print("Index subjektivity vypočítán. Zde jsou výsledky (čím vyšší číslo, tím subjektivnější styl):")
    # Seřadíme od nejvíce subjektivních
    display(articles_df[['title', 'subjectivity_index']].sort_values(by='subjectivity_index', ascending=False).head(10))
else:
    print("Chybí data z POS taggingu pro výpočet indexu.")

### 13. Polarizace přídavných jmen pomocí Word2Vec - S NATVRDO NAPSANÝMI KOTEVNÍMI SLOVY
Pomocí sémantických vektorů určíme, zda jsou přídavná jména v textu spíše popisná (objektivní), nebo hodnotící (subjektivní). Jako kotvy použijeme slova s jasným významem.

In [ ]:
def get_adjective_sentiment_refined(w2v_model, adjectives):
    """Vypočítá sémantickou vzdálenost k subjektivním kotevám vybraným z korpusu."""
    # Nové kotvy vybrané na základě nejčastějších slov v korpusu
    anchors_sub = ['lepší', 'vlastní', 'nutné', 'poslední', 'aktuální', 'rád']
    anchors_obj = ['české', 'vládní', 'státní', 'evropské', 'ekonomické', 'členské']

    valid_sub = [w for w in anchors_sub if w in w2v_model.wv]
    valid_obj = [w for w in anchors_obj if w in w2v_model.wv]

    if not valid_sub or not valid_obj:
        return 0.5

    scores = []
    for adj in adjectives:
        if adj in w2v_model.wv:
            sim_sub = max([w2v_model.wv.similarity(adj, s) for s in valid_sub])
            sim_obj = max([w2v_model.wv.similarity(adj, o) for o in valid_obj])

            adj_score = sim_sub / (sim_sub + sim_obj + 1e-6)
            scores.append(adj_score)

    return round(sum(scores) / len(scores), 4) if scores else 0.5

if 'articles_df' in locals() and 'w2v_model' in locals():
    # Přepočítáme polarizaci s novými kotvami
    articles_df['adj_polarization_score'] = articles_df['adjectives_only'].apply(lambda x: get_adjective_sentiment_refined(w2v_model, x))

    # Přepočítáme finální skóre
    articles_df['refined_bias_score'] = (articles_df['subjectivity_index'] * articles_df['adj_polarization_score'] * 10).round(4)

    print("Zpřesněná analýza s kotvami z korpusu hotova:")
    display(articles_df[['title', 'subjectivity_index', 'adj_polarization_score', 'refined_bias_score']].sort_values(by='refined_bias_score', ascending=False).head(10))
else:
    print("Chybí data nebo model.")

### 14. Extrakce nejčastějších přídavných jmen pro výběr kotev
Projdeme všechny POS tagy a spočítáme četnost přídavných jmen (ADJ), abychom mohli vybrat relevantnější kotevní slova pro náš model.

In [ ]:
from collections import Counter

if 'articles_df' in locals() and 'pos_tags' in articles_df.columns:
    all_adjectives = []
    for tags in articles_df['pos_tags']:
        all_adjectives.extend([word.lower() for word, tag in tags if tag == 'ADJ'])

    most_common_adj = Counter(all_adjectives).most_common(50)

    print("Top 50 nejčastějších přídavných jmen v korpusu:")
    for i, (adj, count) in enumerate(most_common_adj):
        print(f"{i+1}. {adj} ({count}x)", end=' | ' if (i+1) % 5 != 0 else '\n')
else:
    print("Data z POS taggingu nejsou k dispozici.")

### 15. Zpřesněná analýza subjektivity s kotevními slovy z korpusu
Nyní aplikujeme vylepšený výpočet, kde jako referenční body (kotvy) používáme nejčastější slova přímo z analyzovaných článků.

In [ ]:
def get_adjective_sentiment_dynamic(w2v_model, adjectives, common_adj_list):
    """Vylepšený výpočet s dynamickými kotvami z korpusu."""
    # Automaticky rozdělíme top 20 nejčastějších slov na potenciální kotvy
    # Pro demonstraci: slova obsahující 'česk', 'stát', 'evrop' jsou objektivní
    # Slova jako 'lepší', 'vlastní', 'dobrý' jsou subjektivní
    anchors_obj = [word for word, count in common_adj_list if any(x in word for x in ['česk', 'vlád', 'stát', 'evrop', 'ekonom', 'člen'])]
    anchors_sub = [word for word, count in common_adj_list if any(x in word for x in ['lepš', 'vlast', 'nutn', 'posled', 'aktuál', 'rád', 'velk', 'mal'])]

    valid_sub = [w for w in anchors_sub if w in w2v_model.wv]
    valid_obj = [w for w in anchors_obj if w in w2v_model.wv]

    if not valid_sub or not valid_obj:
        return 0.5

    scores = []
    for adj in adjectives:
        if adj in w2v_model.wv:
            sim_sub = np.mean([w2v_model.wv.similarity(adj, s) for s in valid_sub])
            sim_obj = np.mean([w2v_model.wv.similarity(adj, o) for o in valid_obj])

            sim_sub = max(0.001, sim_sub)
            sim_obj = max(0.001, sim_obj)

            adj_score = sim_sub / (sim_sub + sim_obj)
            scores.append(adj_score)

    return round(sum(scores) / len(scores), 4) if scores else 0.5

if 'articles_df' in locals() and 'w2v_model' in locals() and 'most_common_adj' in locals():
    print(f"Používám dynamické objektivní kotvy: { [w for w,c in most_common_adj if any(x in w for x in ['česk', 'vlád', 'stát', 'evrop'])] }")

    articles_df['adj_polarization_corpus'] = articles_df['adjectives_only'].apply(
        lambda x: get_adjective_sentiment_dynamic(w2v_model, x, most_common_adj)
    )

    articles_df['refined_bias_corpus_score'] = (articles_df['subjectivity_index'] * articles_df['adj_polarization_corpus'] * 10).clip(0, 10).round(4)

    print("Dynamické srovnání dokončeno:")
    display(articles_df[['title', 'refined_bias_score', 'refined_bias_corpus_score']].sort_values(by='refined_bias_corpus_score', ascending=False).head(10))
else:
    print("Chybí data, model nebo seznam nejčastějších slov.")

### 16. Detailní srovnání skóre zaujatosti
Porovnání původního skóre (obecné kotvy) a nového skóre (kotvy z korpusu) včetně výpočtu odchylky.

In [ ]:
if 'articles_df' in locals() and 'refined_bias_score' in articles_df.columns and 'refined_bias_corpus_score' in articles_df.columns:
    # Vytvoření kopie pro přehledné zobrazení
    comparison_df = articles_df[['title', 'refined_bias_score', 'refined_bias_corpus_score']].copy()

    # Výpočet rozdílu (pozitivní číslo znamená, že korpusové kotvy zvýšily skóre subjektivity)
    comparison_df['score_diff'] = (comparison_df['refined_bias_corpus_score'] - comparison_df['refined_bias_score']).round(4)

    print("Srovnání: Původní skóre vs. Skóre s kotvami z korpusu")
    display(comparison_df.sort_values(by='refined_bias_corpus_score', ascending=False).head(15))

    # Základní statistika změny
    avg_diff = comparison_df['score_diff'].mean()
    print(f"\nPrůměrná změna ve skóre po zpřesnění: {round(avg_diff, 4)}")
else:
    print("Chybí sloupce se skóre pro srovnání.")

### 17. Srovnání: Neomezená vs. Normalizovaná subjektivita
Zde porovnáme výsledky před a po stabilizaci výpočtu, abychom viděli dopad extrémních sémantických podobností na finální skóre.

In [ ]:
if 'articles_df' in locals() and 'refined_bias_score' in articles_df.columns and 'refined_bias_corpus_score' in articles_df.columns:
    print("--- ANALÝZA BEZ OMEZENÍ (Původní výpočet) ---")
    # Zobrazíme výsledky, kde skóre mohlo být nereálně vysoké
    unnormalized = articles_df[['title', 'refined_bias_score']].sort_values(by='refined_bias_score', ascending=False).head(10)
    display(unnormalized)

    print("\n--- ANALÝZA S NORMALIZACÍ A OMEZENÍM (0-10) ---")
    # Zobrazíme stabilizované výsledky
    normalized = articles_df[['title', 'refined_bias_corpus_score']].sort_values(by='refined_bias_corpus_score', ascending=False).head(10)
    display(normalized)
else:
    print("Chybí data pro srovnání.")

# Návrh od Gemini na vylepšenie

This project is a sophisticated Automated Bias and Subjectivity Analysis Pipeline for Czech news articles. Here is an overview and several ways to expand it for your Text Mining course:

## Current Pipeline Overview

*   **Data Acquisition:** Uses `newspaper3k` and BeautifulSoup to scrape Seznam Zprávy, with custom logic for metadata (authors, dates) and Czech-specific URL filtering.
*   **Preprocessing:** Implements a custom Czech NLP pipeline (lowercase, punctuation/number removal, and a custom stop-word list).
*   **Linguistic Analysis:** Leverages Stanza for POS tagging and lemmatization, specifically extracting adjectives as markers of subjectivity.
*   **Statistical Feature Extraction:** Uses TF-IDF (with bigrams) to identify top keywords and themes.
*   **Semantic Analysis:** Implements a Word2Vec model to calculate semantic distances between words.
*   **Scoring Models:**
    *   **Subjectivity Index:** A ratio of descriptive (ADJ/ADV) vs. factual (NOUN/VERB) words.
    *   **Semic Polarization:** Uses Word2Vec to calculate how 'close' adjectives are to subjective vs. objective 'anchor' words.
    *   **Sentiment Analysis:** Integrates a multilingual BERT model from Hugging Face.

## Expansion Ideas

**1. Comparative Media Analysis (Cross-Source)**
Instead of just Seznam Zprávy, modify the scraper to compare different outlets (e.g., iDNES.cz vs. Deník N vs. Parlamentní listy).
*   **Goal:** Use statistical tests (like T-tests or ANOVA) to see if some domains are statistically more subjective than others.

**2. Named Entity Recognition (NER) & Network Analysis**
Use Stanza's NER to extract Persons, Organizations, and Locations.
*   **Goal:** Build a co-occurrence network. For example, visualize which politicians are most frequently associated with 'negative' sentiment articles vs. 'positive' ones.

**3. Advanced Topic Modeling (LDA or BERTopic)**
Replace the TF-IDF keyword extraction with Latent Dirichlet Allocation (LDA) or BERTopic.
*   **Goal:** Automatically cluster thousands of articles into latent themes (e.g., 'Economy', 'War in Ukraine', 'Domestic Politics') and analyze if certain topics carry higher bias.

**4. Argumentation Mining**
Implement a basic logic to detect 'rhetorical markers' or 'opinionated verbs' (e.g., *tvrdit* vs. *uvést*).
*   **Goal:** Differentiate between a report that 'states facts' and one that 'claims' or 'insinuates'.

**5. Dashboarding with Plotly**
Create interactive visualizations.
*   **Goal:** A scatter plot of Subjectivity vs. Sentiment Score where each point is an article, and hovering reveals the title and top keywords.
